# Snapshot + Hash + Delta Pipeline
## Oracle → ClickHouse — Batch Diário com Detecção Inteligente de Mudanças

---

## Arquitetura

```
ORACLE (Fonte Volátil)
 ↓
 ↓ Full Extract D-1 (Batch Diário)
 ↓

 BRONZE — Snapshot Bruto 
 • Full extract diário 
 • Dados brutos sem transformação 
 • Idempotente por ref_date 
 • Formato: Parquet particionado 

 
 Spark Transformation
 ↓

 SILVER — Normalização + Hash 
 • Normalização de tipos e valores 
 • Primary Key lógica definida 
 • row_hash SHA-256 calculado 
 • PARTITION BY ref_date 
 • ORDER BY (ref_date, pk) 

 
 Comparação D-1 vs D-2
 ↓

 GOLD — Deltas (INSERT/UPDATE/DELETE) 
 • Comparação de snapshots 
 • Identificação de mudanças 
 • Auditoria completa 
 • ClickHouse = Fonte de Verdade 

```

---

## Princípios

1. **Sem CDC**: Oracle não tem logs confiáveis
2. **Batch Diário**: Full extract D-1
3. **Idempotência**: Reprocessamento seguro
4. **Hash por Linha**: Comparação eficiente
5. **ClickHouse = Verdade**: Sistema de memória histórica

---

## Detecção de Mudanças

| Operação | Condição | Lógica |
|----------|----------|--------|
| **INSERT** | PK existe em D-1 e não existe em D-2 | Novo registro |
| **UPDATE** | PK existe em ambos, mas `row_hash` é diferente | Registro modificado |
| **DELETE** | PK existe em D-2 e não existe em D-1 | Registro removido |
| **NO_CHANGE** | PK existe em ambos e `row_hash` é igual | Sem mudança |

---
## 1. Setup e Configuração

In [21]:
import warnings
warnings.filterwarnings('ignore')

import os
from pathlib import Path
from datetime import datetime, timedelta, date
from typing import List, Dict, Any, Optional
import json

from pyspark.sql import SparkSession, DataFrame as SparkDataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import clickhouse_connect
import pandas as pd
from dotenv import load_dotenv
import psutil

load_dotenv()


def get_memory_mb():
    try:
        return round(psutil.Process().memory_info().rss / (1024 * 1024), 2)
    except Exception:
        return 0.0


print("Imports carregados com sucesso")

Imports carregados com sucesso


In [22]:
CH_HOST = os.getenv('CLICKHOUSE_HOST', 'e1a1lieug8.us-central1.gcp.clickhouse.cloud')
CH_PORT = int(os.getenv('CLICKHOUSE_PORT', 8443))
CH_USER = os.getenv('CLICKHOUSE_USER', 'default')
CH_PASSWORD = os.getenv('CLICKHOUSE_PASSWORD', '_uv765EvWphL_')

CH_DB_BRONZE = 'raw'
CH_DB_SILVER = 'trusted'
CH_DB_GOLD = 'gold'

JDBC_URL_BRONZE = f"jdbc:clickhouse:https://{CH_HOST}:{CH_PORT}/{CH_DB_BRONZE}?ssl=true"
JDBC_URL_SILVER = f"jdbc:clickhouse:https://{CH_HOST}:{CH_PORT}/{CH_DB_SILVER}?ssl=true"
JDBC_URL_GOLD = f"jdbc:clickhouse:https://{CH_HOST}:{CH_PORT}/{CH_DB_GOLD}?ssl=true"

print(f"ClickHouse Host: {CH_HOST}")
print(f"Bronze DB: {CH_DB_BRONZE}")
print(f"Silver DB: {CH_DB_SILVER}")
print(f"Gold DB: {CH_DB_GOLD}")

ClickHouse Host: e1a1lieug8.us-central1.gcp.clickhouse.cloud
Bronze DB: raw
Silver DB: trusted
Gold DB: gold


In [23]:
print("Inicializando Spark com configuracoes otimizadas...\n")

spark = SparkSession.builder \
.appName("SnapshotHashDelta-Pipeline") \
    .config("spark.jars.packages", "com.clickhouse:clickhouse-jdbc:0.4.6,com.clickhouse:clickhouse-client:0.4.6") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "100") \
    .config("spark.default.parallelism", "50") \
    .config("spark.sql.autoBroadcastJoinThreshold", "50MB") \
    .config("spark.memory.fraction", "0.8") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("Spark inicializado")
print(f" Versão: {spark.version}")
print(f" Paralelismo: {spark.sparkContext.defaultParallelism}")
print(f" Shuffle Partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")

Inicializando Spark com configuracoes otimizadas...

Spark inicializado
 Versão: 3.4.1
 Paralelismo: 50
 Shuffle Partitions: 100


In [24]:
print(f"Conectando ao ClickHouse: {CH_HOST}:{CH_PORT}")

client = clickhouse_connect.get_client(
    host=CH_HOST,
    port=CH_PORT,
    username=CH_USER,
    password=CH_PASSWORD,
    secure=True
)

version = client.query("SELECT version()").result_rows[0][0]
print(f"ClickHouse conectado")
print(f" Versão: {version}")

# Criar databases se não existirem
for db in [CH_DB_BRONZE, CH_DB_SILVER, CH_DB_GOLD]:
    client.command(f"CREATE DATABASE IF NOT EXISTS {db}")
    print(f" OK Database {db} pronto")

Conectando ao ClickHouse: e1a1lieug8.us-central1.gcp.clickhouse.cloud:8443
ClickHouse conectado
 Versão: 25.10.1.7375
 OK Database raw pronto
 OK Database trusted pronto
 OK Database gold pronto


---
## 2. CAMADA BRONZE — Snapshot Bruto

### Objetivo
- Realizar **full extract diário** (D-1) do Oracle
- Persistir dados **brutos** sem transformação
- Garantir **idempotência** por `ref_date`
- Base para reprocessamento seguro

In [25]:
def extract_bronze_snapshot(
        table_name: str,
        ref_date: date,
        source_df: SparkDataFrame = None,
        overwrite: bool = True
) -> Dict[str, Any]:
    """
    Extrai snapshot bruto diário do Oracle para camada Bronze.
 
    Args:
    table_name: Nome da tabela fonte (ex: 'SF2030')
    ref_date: Data de referência do snapshot (geralmente D-1)
    source_df: DataFrame Spark com dados do Oracle (simulação)
    overwrite: Se True, sobrescreve snapshot existente para mesma data
 
    Returns:
    Dict com estatísticas da extração
    """
    start_time = datetime.now()
    memory_start = get_memory_mb()

    print(f"\n{'=' * 80}")
    print(f" BRONZE SNAPSHOT EXTRACTION")
    print(f"{'=' * 80}")
    print(f"Tabela: {table_name}")
    print(f"Data Referência: {ref_date}")
    print(f" Modo: {'OVERWRITE' if overwrite else 'APPEND'}")

    # Simular leitura do Oracle (na prática seria via JDBC)
    if source_df is None:
        print("\n Nenhum DataFrame fornecido, usando Silver como fonte de simulação")
        # Ler da Silver como simulação
        source_df = spark.read.jdbc(
            url=JDBC_URL_SILVER,
            table=f"{CH_DB_SILVER}.{table_name.lower()}",
            properties={
                "driver": "com.clickhouse.jdbc.ClickHouseDriver",
                "user": CH_USER,
                "password": CH_PASSWORD,
                "ssl": "true"
            }
        )

    # Remover colunas técnicas da Silver
    tech_cols = ['_bronze_ingestion_timestamp', '_silver_ingestion_timestamp',
                 '_silver_processing_date', '_data_quality_flag']
    for col in tech_cols:
        if col in source_df.columns:
            source_df = source_df.drop(col)

    # Adicionar metadados Bronze
    df_bronze = source_df.withColumn(
        "_ref_date", F.lit(ref_date)
    ).withColumn(
        "_bronze_ingestion_timestamp", F.current_timestamp()
    )

    row_count = df_bronze.count()
    print(f"\nRegistros extraídos: {row_count:,}")

    # Preparar tabela Bronze no ClickHouse
    bronze_table = f"{table_name.lower()}_bronze_snapshot"

    if overwrite:
        # Deletar snapshot existente para mesma data (idempotência)
        try:
            client.command(f"""
            ALTER TABLE {CH_DB_BRONZE}.{bronze_table}
            DELETE WHERE _ref_date = '{ref_date}'
             """)
            print(f" Snapshot anterior removido para {ref_date}")
        except:
            pass  # Tabela ainda não existe

    # Criar tabela Bronze se não existir
    # Gerar DDL dinamicamente
    column_defs = []
    for field in df_bronze.schema.fields:
        spark_type = field.dataType
        if field.name == "_ref_date":
            ch_type = "Date"
        elif field.name == "_bronze_ingestion_timestamp":
            ch_type = "DateTime64(3)"
        elif isinstance(spark_type, StringType):
            ch_type = "Nullable(String)"
        elif isinstance(spark_type, IntegerType):
            ch_type = "Nullable(Int32)"
        elif isinstance(spark_type, LongType):
            ch_type = "Nullable(Int64)"
        elif isinstance(spark_type, DoubleType):
            ch_type = "Nullable(Float64)"
        elif isinstance(spark_type, DateType):
            ch_type = "Nullable(Date)"
        elif isinstance(spark_type, TimestampType):
            ch_type = "Nullable(DateTime64(3))"
        else:
            ch_type = "Nullable(String)"
        column_defs.append(f"`{field.name}` {ch_type}")

    columns_ddl = ",\n ".join(column_defs)

    create_ddl = f"""
    CREATE TABLE IF NOT EXISTS {CH_DB_BRONZE}.{bronze_table} (
    {columns_ddl}
    )
    ENGINE = MergeTree()
    PARTITION BY _ref_date
    ORDER BY (_ref_date, _bronze_ingestion_timestamp)
    """

    client.command(create_ddl)
    print(f"Tabela Bronze criada/verificada: {bronze_table}")

    # Gravar no ClickHouse
    print(f"\nGravando snapshot Bronze...")
    df_bronze.write.jdbc(
        url=JDBC_URL_BRONZE,
        table=f"{CH_DB_BRONZE}.{bronze_table}",
        mode="append",
        properties={
            "driver": "com.clickhouse.jdbc.ClickHouseDriver",
            "user": CH_USER,
            "password": CH_PASSWORD,
            "ssl": "true",
            "batchsize": "100000"
        }
    )

    duration = (datetime.now() - start_time).total_seconds()
    throughput = row_count / duration if duration > 0 else 0
    memory_end = get_memory_mb()

    print(f"\nBRONZE SNAPSHOT CONCLUÍDO")
    print(f" Registros: {row_count:,}")
    print(f" Duracao: {duration:.2f}s")
    print(f" Throughput: {throughput:,.0f} rows/s")
    print(f" Memoria (RSS): {memory_start:.0f} MB -> {memory_end:.0f} MB (delta {memory_end - memory_start:+.0f} MB)")
    print(f" Localização: {CH_DB_BRONZE}.{bronze_table}")
    print(f"{'=' * 80}\n")

    return {
        'table_name': bronze_table,
        'ref_date': ref_date,
        'row_count': row_count,
        'duration_seconds': duration,
        'throughput': throughput,
        'memory_mb_start': memory_start,
        'memory_mb_end': memory_end,
        'status': 'success'
    }


print("Função extract_bronze_snapshot definida")

Função extract_bronze_snapshot definida


---
## 3. CAMADA SILVER — Normalização + Hash SHA-256

### Objetivo
- Normalizar tipos e valores (trim, cast, null-safe)
- Definir **Primary Key lógica**
- Calcular **row_hash SHA-256** de todas as colunas de negócio
- Preparar para comparação eficiente

In [26]:
def process_silver_snapshot(
    table_name: str,
    ref_date: date,
    primary_keys: List[str],
    business_columns: Optional[List[str]] = None
) -> Dict[str, Any]:
    """
    Processa snapshot Bronze → Silver com normalização e hash.

    Args:
    table_name: Nome da tabela (ex: 'SF2030')
    ref_date: Data de referência do snapshot
    primary_keys: Lista de colunas que compõem a PK lógica
    business_columns: Colunas de negócio para hash (None = todas exceto PK)

    Returns:
    Dict com estatísticas do processamento
    """
    start_time = datetime.now()
    memory_start = get_memory_mb()

    print(f"{'='*80}")
    print(f"SILVER SNAPSHOT PROCESSING")
    print(f"{'='*80}")
    print(f"Tabela: {table_name}")
    print(f"Data Referência: {ref_date}")
    print(f" Primary Keys: {', '.join(primary_keys)}")

    # Ler snapshot Bronze
    bronze_table = f"{table_name.lower()}_bronze_snapshot"

    print(f" Lendo Bronze: {bronze_table}...")
    df_bronze = spark.read.jdbc(
        url=JDBC_URL_BRONZE,
        table=f"{CH_DB_BRONZE}.{bronze_table}",
        properties={
            "driver": "com.clickhouse.jdbc.ClickHouseDriver",
            "user": CH_USER,
            "password": CH_PASSWORD,
            "ssl": "true"
        }
    ).filter(F.col("_ref_date") == F.lit(ref_date))

    row_count_bronze = df_bronze.count()
    print(f" OK Registros Bronze: {row_count_bronze:,}")

    # Normalização
    print(f"Aplicando normalização...")

    # Identificar colunas de negócio
    technical_cols = ['_ref_date', '_bronze_ingestion_timestamp']
    all_cols = [c for c in df_bronze.columns if c not in technical_cols]

    if business_columns is None:
        business_columns = all_cols

    print(f" Colunas de negócio: {len(business_columns)}")

    # Normalizar strings (trim, uppercase)
    df_normalized = df_bronze
    for col in business_columns:
        if dict(df_bronze.dtypes)[col] == 'string':
            df_normalized = df_normalized.withColumn(
                col,
                F.trim(F.upper(F.col(col)))
            )

    # Calcular row_hash (SHA-256)
    print(f" Calculando row_hash (SHA-256)...")

    # Concatenar todas as colunas de negócio ordenadas
    hash_expression = F.concat_ws(
        "|",
        *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in sorted(business_columns)]
    )

    df_silver = df_normalized.withColumn(
        "row_hash",
        F.sha2(hash_expression, 256)
    ).withColumn(
        "ref_date",
        F.col("_ref_date")
    ).withColumn(
        "_silver_processing_timestamp",
        F.current_timestamp()
    )

    # Selecionar colunas finais (PK + business + técnicas)
    final_columns = primary_keys + business_columns + ['ref_date', 'row_hash', '_silver_processing_timestamp']
    final_columns = list(dict.fromkeys(final_columns))  # Remove duplicatas mantendo ordem

    df_silver = df_silver.select(*final_columns)

    # Remover duplicatas pela PK (última ocorrência)
    window_spec = Window.partitionBy(*primary_keys).orderBy(F.desc("_silver_processing_timestamp"))
    df_silver = df_silver.withColumn("row_num", F.row_number().over(window_spec)) \
        .filter(F.col("row_num") == 1) \
        .drop("row_num")

    row_count_silver = df_silver.count()
    duplicates = row_count_bronze - row_count_silver

    print(f" OK Registros após deduplicação: {row_count_silver:,}")
    if duplicates > 0:
        print(f" Duplicatas removidas: {duplicates:,}")

    # Criar/atualizar tabela Silver
    silver_table = f"{table_name.lower()}_silver_snapshot"

    # Deletar snapshot existente para mesma data (idempotência)
    exists_q = f"""
    SELECT count()
    FROM system.tables
    WHERE database = '{CH_DB_SILVER}' AND name = '{silver_table}'
    """
    if client.query(exists_q).result_rows[0][0] > 0:
        client.command(f"""
        ALTER TABLE {CH_DB_SILVER}.{silver_table}
        DELETE WHERE ref_date = '{ref_date}'
        """)
        print(f" Snapshot anterior removido para {ref_date}")

    # Criar tabela se não existir
    column_defs = []
    for field in df_silver.schema.fields:
        spark_type = field.dataType

        if isinstance(spark_type, StringType):
            ch_type = "Nullable(String)"
        elif isinstance(spark_type, IntegerType):
            ch_type = "Nullable(Int32)"
        elif isinstance(spark_type, LongType):
            ch_type = "Nullable(Int64)"
        elif isinstance(spark_type, DoubleType):
            ch_type = "Nullable(Float64)"
        elif isinstance(spark_type, DateType):
            ch_type = "Nullable(Date)"
        elif isinstance(spark_type, TimestampType):
            ch_type = "Nullable(DateTime64(3))"
        else:
            ch_type = "Nullable(String)"

        column_defs.append(f"`{field.name}` {ch_type}")

    columns_ddl = ",\n    ".join(column_defs)
    pk_order = ", ".join(primary_keys)

    create_ddl = f"""
    CREATE TABLE IF NOT EXISTS {CH_DB_SILVER}.{silver_table} (
    {columns_ddl}
    )
    ENGINE = MergeTree()
    PARTITION BY ref_date
    ORDER BY (ref_date, {pk_order})
    """

    client.command(create_ddl)
    print(f"Tabela Silver criada/verificada: {silver_table}")

    # Gravar no ClickHouse
    print(f"Gravando snapshot Silver...")
    df_silver.write.jdbc(
        url=JDBC_URL_SILVER,
        table=f"{CH_DB_SILVER}.{silver_table}",
        mode="append",
        properties={
            "driver": "com.clickhouse.jdbc.ClickHouseDriver",
            "user": CH_USER,
            "password": CH_PASSWORD,
            "ssl": "true",
            "batchsize": "100000"
        }
    )

    duration = (datetime.now() - start_time).total_seconds()
    throughput = row_count_silver / duration if duration > 0 else 0
    memory_end = get_memory_mb()

    print(f"SILVER SNAPSHOT CONCLUÍDO")
    print(f" Registros: {row_count_silver:,}")
    print(f" Duracao: {duration:.2f}s")
    print(f" Throughput: {throughput:,.0f} rows/s")
    print(f" Memoria (RSS): {memory_start:.0f} MB -> {memory_end:.0f} MB (delta {memory_end - memory_start:+.0f} MB)")
    print(f" Localização: {CH_DB_SILVER}.{silver_table}")
    print(f" Hash: SHA-256 de {len(business_columns)} colunas")
    print(f"{'='*80}\n")

    return {
        'table_name': silver_table,
        'ref_date': ref_date,
        'row_count': row_count_silver,
        'duplicates_removed': duplicates,
        'duration_seconds': duration,
        'throughput': throughput,
        'memory_mb_start': memory_start,
        'memory_mb_end': memory_end,
        'status': 'success'
    }

print("Função process_silver_snapshot definida")

Função process_silver_snapshot definida


---
## 4. CAMADA GOLD — Comparação de Snapshots e Deltas

### Objetivo
- Comparar snapshot D-1 vs D-2
- Identificar INSERT, UPDATE, DELETE
- Gerar tabela de deltas auditável
- ClickHouse como fonte de verdade histórica

In [27]:
def compute_delta(
    table_name: str,
    ref_date_current: date,
    ref_date_previous: date,
    primary_keys: List[str]
) -> Dict[str, Any]:
    """
    Compara dois snapshots Silver consecutivos e identifica deltas.
 
    Args:
    table_name: Nome da tabela (ex: 'SF2030')
    ref_date_current: Data do snapshot atual (D-1)
    ref_date_previous: Data do snapshot anterior (D-2)
    primary_keys: Lista de colunas da PK
 
    Returns:
    Dict com estatísticas dos deltas
    """
    start_time = datetime.now()
    memory_start = get_memory_mb()
 
    print(f"\n{'='*80}")
    print(f"GOLD DELTA COMPUTATION")
    print(f"{'='*80}")
    print(f"Tabela: {table_name}")
    print(f"Current (D-1): {ref_date_current}")
    print(f"Previous (D-2): {ref_date_previous}")
    print(f" Primary Keys: {', '.join(primary_keys)}")
 
    silver_table = f"{table_name.lower()}_silver_snapshot"
 
    # Ler snapshots Silver
    print(f"\n Lendo snapshots Silver...")
 
    df_current = spark.read.jdbc(
        url=JDBC_URL_SILVER,
        table=f"{CH_DB_SILVER}.{silver_table}",
        properties={
            "driver": "com.clickhouse.jdbc.ClickHouseDriver",
            "user": CH_USER,
            "password": CH_PASSWORD,
            "ssl": "true"
        }
    ).filter(F.col("ref_date") == F.lit(ref_date_current))
 
    df_previous = spark.read.jdbc(
        url=JDBC_URL_SILVER,
        table=f"{CH_DB_SILVER}.{silver_table}",
        properties={
            "driver": "com.clickhouse.jdbc.ClickHouseDriver",
            "user": CH_USER,
            "password": CH_PASSWORD,
            "ssl": "true"
        }
    ).filter(F.col("ref_date") == F.lit(ref_date_previous))
 
    count_current = df_current.count()
    count_previous = df_previous.count()
 
    print(f" OK Snapshot Current: {count_current:,} registros")
    print(f" OK Snapshot Previous: {count_previous:,} registros")
 
    # Preparar para comparação
    pk_cols = primary_keys
 
    # Criar views temporárias com PK + hash
    df_current_keys = df_current.select(*pk_cols, "row_hash").withColumnRenamed("row_hash", "hash_current")
    df_previous_keys = df_previous.select(*pk_cols, "row_hash").withColumnRenamed("row_hash", "hash_previous")
 
    print(f"\nIdentificando mudanças...")
 
    # Full outer join nas PKs
    df_comparison = df_current_keys.join(
        df_previous_keys,
        on=pk_cols,
        how="full_outer"
    )
 
    # Identificar operação
    df_deltas = df_comparison.withColumn(
        "operation",
        F.when(
            (F.col("hash_current").isNotNull()) & (F.col("hash_previous").isNull()),
            F.lit("I")  # INSERT
        ).when(
            (F.col("hash_current").isNull()) & (F.col("hash_previous").isNotNull()),
            F.lit("D")  # DELETE
        ).when(
            (F.col("hash_current").isNotNull()) & 
            (F.col("hash_previous").isNotNull()) & 
            (F.col("hash_current") != F.col("hash_previous")),
            F.lit("U")  # UPDATE
        ).otherwise(F.lit("N"))  # NO_CHANGE
    ).filter(
        F.col("operation") != "N"  # Ignorar registros sem mudança
    )
 
    # Adicionar dados completos para INSERT e UPDATE
    # (JOIN com snapshot current para trazer todas as colunas)
    df_deltas_full = df_deltas.join(
        df_current,
        on=pk_cols,
        how="left"
    ).withColumn(
        "delta_date",
        F.lit(ref_date_current)
    ).withColumn(
        "delta_timestamp",
        F.current_timestamp()
    )
 
    # Contar por operação
    delta_counts = df_deltas_full.groupBy("operation").count().collect()
    delta_stats = {row['operation']: row['count'] for row in delta_counts}
 
    inserts = delta_stats.get('I', 0)
    updates = delta_stats.get('U', 0)
    deletes = delta_stats.get('D', 0)
    total_deltas = inserts + updates + deletes
 
    print(f"\nDELTAS IDENTIFICADOS:")
    print(f" INSERT: {inserts:,}")
    print(f" UPDATE: {updates:,}")
    print(f" DELETE: {deletes:,}")
    print(f" TOTAL: {total_deltas:,}")
 
    if total_deltas == 0:
        print(f"\nNenhuma mudanca detectada entre {ref_date_previous} e {ref_date_current}")
        return {
            'table_name': table_name,
            'ref_date_current': ref_date_current,
            'ref_date_previous': ref_date_previous,
            'inserts': 0,
            'updates': 0,
            'deletes': 0,
            'total_deltas': 0,
            'status': 'no_changes'
        }
 
    # Criar tabela Gold de deltas
    delta_table = f"{table_name.lower()}_gold_deltas"
 
    # Deletar deltas existentes para mesma data (idempotência)
    client.command(f"""
    ALTER TABLE IF EXISTS {CH_DB_GOLD}.{delta_table}
    DELETE WHERE delta_date = '{ref_date_current}'
    """)
    print(f"\n Deltas anteriores removidos para {ref_date_current}")
 
    # Criar tabela de deltas
    column_defs = []
    for field in df_deltas_full.schema.fields:
        if field.name in ['hash_current', 'hash_previous', 'ref_date', '_silver_processing_timestamp']:
            continue
 
        spark_type = field.dataType
 
        if isinstance(spark_type, StringType):
            ch_type = "Nullable(String)"
        elif isinstance(spark_type, IntegerType):
            ch_type = "Nullable(Int32)"
        elif isinstance(spark_type, LongType):
            ch_type = "Nullable(Int64)"
        elif isinstance(spark_type, DoubleType):
            ch_type = "Nullable(Float64)"
        elif isinstance(spark_type, DateType):
            ch_type = "Nullable(Date)"
        elif isinstance(spark_type, TimestampType):
            ch_type = "Nullable(DateTime64(3))"
        else:
            ch_type = "Nullable(String)"
 
        column_defs.append(f"`{field.name}` {ch_type}")
 
    columns_ddl = ",\n ".join(column_defs)
    pk_order = ", ".join(primary_keys)
 
    create_ddl = f"""
    CREATE TABLE IF NOT EXISTS {CH_DB_GOLD}.{delta_table} (
    {columns_ddl}
    )
    ENGINE = MergeTree()
    PARTITION BY delta_date
    ORDER BY (delta_date, operation, {pk_order})
    """
 
    client.command(create_ddl)
    print(f"Tabela Gold criada/verificada: {delta_table}")

    # Gravar deltas
    print(f"\nGravando deltas Gold...")
 
    # Selecionar apenas colunas relevantes
    cols_to_write = [c for c in df_deltas_full.columns 
                     if c not in ['hash_current', 'hash_previous', 'ref_date', '_silver_processing_timestamp']]
 
    df_deltas_full.select(*cols_to_write).write.jdbc(
        url=JDBC_URL_GOLD,
        table=f"{CH_DB_GOLD}.{delta_table}",
        mode="append",
        properties={
            "driver": "com.clickhouse.jdbc.ClickHouseDriver",
            "user": CH_USER,
            "password": CH_PASSWORD,
            "ssl": "true",
            "batchsize": "100000"
        }
    )
 
    duration = (datetime.now() - start_time).total_seconds()
    memory_end = get_memory_mb()
 
    print(f"\nGOLD DELTAS CONCLUÍDO")
    print(f" Total Deltas: {total_deltas:,}")
    print(f" Duracao: {duration:.2f}s")
    print(f" Memoria (RSS): {memory_start:.0f} MB -> {memory_end:.0f} MB (delta {memory_end - memory_start:+.0f} MB)")
    print(f" Localização: {CH_DB_GOLD}.{delta_table}")
    print(f"{'='*80}\n")
 
    return {
        'table_name': delta_table,
        'ref_date_current': ref_date_current,
        'ref_date_previous': ref_date_previous,
        'inserts': inserts,
        'updates': updates,
        'deletes': deletes,
        'total_deltas': total_deltas,
        'duration_seconds': duration,
        'memory_mb_start': memory_start,
        'memory_mb_end': memory_end,
        'status': 'success'
    }

print("Função compute_delta definida")

Função compute_delta definida


---
## 5. Pipeline Completo End-to-End

In [28]:
def run_snapshot_delta_pipeline(
    table_name: str,
    ref_date: date,
    primary_keys: List[str],
    source_df: SparkDataFrame = None,
    compute_deltas: bool = True
) -> Dict[str, Any]:
    """
    Executa pipeline completo: Bronze → Silver → Gold.
 
    Args:
    table_name: Nome da tabela
    ref_date: Data de referência (D-1)
    primary_keys: Colunas da PK
    source_df: DataFrame fonte (opcional)
    compute_deltas: Se True, calcula deltas vs dia anterior
 
    Returns:
    Dict com resultado completo
    """
    pipeline_start = datetime.now()
 
    print(f"\n{'='*80}")
    print(f"SNAPSHOT + HASH + DELTA PIPELINE")
    print(f"{'='*80}")
    print(f"Tabela: {table_name}")
    print(f"Data: {ref_date}")
    print(f" PK: {', '.join(primary_keys)}")
    print(f"{'='*80}")
 
    results = {}
 
    # BRONZE
    try:
        bronze_result = extract_bronze_snapshot(
            table_name=table_name,
            ref_date=ref_date,
            source_df=source_df
        )
        results['bronze'] = bronze_result
    except Exception as e:
        print(f"Erro no Bronze: {e}")
        return {'status': 'failed', 'stage': 'bronze', 'error': str(e)}
 
    # SILVER
    try:
        silver_result = process_silver_snapshot(
            table_name=table_name,
            ref_date=ref_date,
            primary_keys=primary_keys
        )
        results['silver'] = silver_result
    except Exception as e:
        print(f"Erro no Silver: {e}")
        return {'status': 'failed', 'stage': 'silver', 'error': str(e)}
 
    # GOLD (Deltas)
    if compute_deltas:
        try:
            ref_date_previous = ref_date - timedelta(days=1)
 
            # Verificar se existe snapshot anterior
            silver_table = f"{table_name.lower()}_silver_snapshot"
            check_query = f"""
            SELECT count(*) as cnt 
            FROM {CH_DB_SILVER}.{silver_table}
            WHERE ref_date = '{ref_date_previous}'
            """
 
            count_previous = client.query(check_query).result_rows[0][0]
 
            if count_previous > 0:
                delta_result = compute_delta(
                    table_name=table_name,
                    ref_date_current=ref_date,
                    ref_date_previous=ref_date_previous,
                    primary_keys=primary_keys
                )
                results['gold'] = delta_result
            else:
                print(f"\nSnapshot anterior nao encontrado ({ref_date_previous})")
                print(f" Deltas não serão calculados (primeiro snapshot)\n")
                results['gold'] = {'status': 'skipped', 'reason': 'no_previous_snapshot'}
        except Exception as e:
            print(f"Erro no Gold: {e}")
            results['gold'] = {'status': 'failed', 'error': str(e)}
 
    pipeline_duration = (datetime.now() - pipeline_start).total_seconds()
 
    print(f"\n{'='*80}")
    print(f"PIPELINE COMPLETO")
    print(f"{'='*80}")
    print(f"Duracao Total: {pipeline_duration:.2f}s")
    print(f"{'='*80}\n")
 
    results['pipeline_duration'] = pipeline_duration
    results['status'] = 'success'
 
    return results

print("Função run_snapshot_delta_pipeline definida")

Função run_snapshot_delta_pipeline definida


---
## 6. Exemplo Prático — Tabela SF2030

Vamos simular 2 dias de snapshots e detectar mudanças.

In [29]:
# Definir configuração da tabela SF2
TABLE_NAME = 'SF2030'
PRIMARY_KEYS = ['F2_DOC', 'F2_SERIE', 'F2_CLIENTE', 'F2_LOJA']

# Data de referência: ontem (D-1)
today = date.today()
ref_date_d1 = today - timedelta(days=1)
ref_date_d2 = today - timedelta(days=2)

print(f"Hoje: {today}")
print(f"D-1 (ref_date): {ref_date_d1}")
print(f"D-2 (ref_date): {ref_date_d2}")

Hoje: 2026-02-09
D-1 (ref_date): 2026-02-08
D-2 (ref_date): 2026-02-07


In [ ]:
# Executar pipeline para D-2 (dia anterior para comparação)
print("\n" + "="*80)
print("CRIANDO SNAPSHOT D-2 (Baseline)")
print("="*80 + "\n")

result_d2 = run_snapshot_delta_pipeline(
    table_name=TABLE_NAME,
    ref_date=ref_date_d2,
    primary_keys=PRIMARY_KEYS,
    source_df=df_simulado_d2,  # Passar DataFrame simulado
    compute_deltas=False  # Primeiro snapshot, sem deltas
)

print(f"\nSnapshot D-2 criado com sucesso")

In [ ]:
# Executar pipeline para D-1 (dia atual) e calcular deltas
print("\n" + "="*80)
print("CRIANDO SNAPSHOT D-1 + CALCULANDO DELTAS")
print("="*80 + "\n")

result_d1 = run_snapshot_delta_pipeline(
    table_name=TABLE_NAME,
    ref_date=ref_date_d1,
    primary_keys=PRIMARY_KEYS,
    source_df=df_simulado_d1,  # Passar DataFrame simulado D-1
    compute_deltas=True  # Calcular deltas vs D-2
)

print(f"\nSnapshot D-1 e deltas calculados com sucesso")

In [33]:
# Executar pipeline para D-1 (dia atual) e calcular deltas
print("\n" + "="*80)
print("CRIANDO SNAPSHOT D-1 + CALCULANDO DELTAS")
print("="*80 + "\n")

result_d1 = run_snapshot_delta_pipeline(
    table_name=TABLE_NAME,
    ref_date=ref_date_d1,
    primary_keys=PRIMARY_KEYS,
    compute_deltas=True # Calcular deltas vs D-2
)

print(f"\nSnapshot D-1 e deltas calculados com sucesso")


CRIANDO SNAPSHOT D-1 + CALCULANDO DELTAS


SNAPSHOT + HASH + DELTA PIPELINE
Tabela: SF2030
Data: 2026-02-08
 PK: F2_DOC, F2_SERIE, F2_CLIENTE, F2_LOJA

 BRONZE SNAPSHOT EXTRACTION
Tabela: SF2030
Data Referência: 2026-02-08
 Modo: OVERWRITE

 Nenhum DataFrame fornecido, usando Silver como fonte de simulação
Erro no Bronze: An error occurred while calling o70.jdbc.
: java.sql.BatchUpdateException: Code: 60. DB::Exception: Unknown table expression identifier 'trusted.sf2030' in scope SELECT * FROM trusted.sf2030 WHERE 1 = 0. (UNKNOWN_TABLE) (version 25.10.1.7375 (official build))
, server ClickHouseNode [uri=https://e1a1lieug8.us-central1.gcp.clickhouse.cloud:8443/trusted, options={sslmode=STRICT}]@287291726
	at com.clickhouse.jdbc.SqlExceptionUtils.batchUpdateError(SqlExceptionUtils.java:107)
	at com.clickhouse.jdbc.internal.SqlBasedPreparedStatement.executeAny(SqlBasedPreparedStatement.java:219)
	at com.clickhouse.jdbc.internal.SqlBasedPreparedStatement.executeQuery(SqlBasedPreparedSta

---
## 7. Análise e Validação

In [32]:
# Visualizar deltas Gold
delta_table = f"{TABLE_NAME.lower()}_gold_deltas"

deltas_summary = client.query_df(f"""
    SELECT 
    delta_date,
    operation,
    count(*) as qty
    FROM {CH_DB_GOLD}.{delta_table}
    GROUP BY delta_date, operation
    ORDER BY delta_date DESC, operation
""")

print("\n" + "="*80)
print("RESUMO DE DELTAS")
print("="*80)
print(deltas_summary.to_string(index=False))
print("="*80)

DatabaseError: :HTTPDriver for https://e1a1lieug8.us-central1.gcp.clickhouse.cloud:8443 returned response code 404)
 Code: 60. DB::Exception: Unknown table expression identifier 'gold.sf2030_gold_deltas' in scope SELECT delta_date, operation, count(*) AS qty FROM gold.sf2030_gold_deltas GROUP BY delta_date, operation ORDER BY delta_date DESC, operation ASC. (UNKNOWN_TABLE) (version 25.10.1.7375 (official build))


In [ ]:
# Visualizar alguns exemplos de cada operação
print("\n EXEMPLOS DE INSERT:")
inserts = client.query_df(f"""
    SELECT *
    FROM {CH_DB_GOLD}.{delta_table}
    WHERE operation = 'I'
    LIMIT 3
""")
print(inserts.to_string(index=False) if len(inserts) > 0 else " Nenhum INSERT encontrado")

print("\n EXEMPLOS DE UPDATE:")
updates = client.query_df(f"""
    SELECT *
    FROM {CH_DB_GOLD}.{delta_table}
    WHERE operation = 'U'
    LIMIT 3
""")
print(updates.to_string(index=False) if len(updates) > 0 else " Nenhum UPDATE encontrado")

print("\n EXEMPLOS DE DELETE:")
deletes = client.query_df(f"""
    SELECT *
    FROM {CH_DB_GOLD}.{delta_table}
    WHERE operation = 'D'
    LIMIT 3
""")
print(deletes.to_string(index=False) if len(deletes) > 0 else " Nenhum DELETE encontrado")

In [ ]:
# Validar integridade: comparar contagens
silver_table = f"{TABLE_NAME.lower()}_silver_snapshot"

validation = client.query_df(f"""
    WITH snapshot_counts AS (
        SELECT 
            ref_date,
            count(*) as total_rows
        FROM {CH_DB_SILVER}.{silver_table}
        WHERE ref_date IN ('{ref_date_d1}', '{ref_date_d2}')
        GROUP BY ref_date
    ),
    delta_summary AS (
        SELECT
            sumIf(1, operation = 'I') as inserts,
            sumIf(1, operation = 'U') as updates,
            sumIf(1, operation = 'D') as deletes
        FROM {CH_DB_GOLD}.{delta_table}
        WHERE delta_date = '{ref_date_d1}'
    )
    SELECT 
        s1.total_rows as d1_total,
        s2.total_rows as d2_total,
        d.inserts,
        d.updates,
        d.deletes,
        (s2.total_rows + d.inserts - d.deletes) as expected_d1,
        s1.total_rows = (s2.total_rows + d.inserts - d.deletes) as is_valid
    FROM snapshot_counts s1
    CROSS JOIN snapshot_counts s2
    CROSS JOIN delta_summary d
    WHERE s1.ref_date = '{ref_date_d1}'
    AND s2.ref_date = '{ref_date_d2}'
""")

print("\n" + "="*80)
print("VALIDAÇÃO DE INTEGRIDADE")
print("="*80)
print(validation.to_string(index=False))
print("="*80)

if validation['is_valid'].iloc[0]:
    print("\nVALIDAÇÃO PASSOU: D2 + Inserts - Deletes = D1")
else:
    print("\n VALIDAÇÃO FALHOU: Inconsistência detectada!")

---
## 8. Resumo da Arquitetura Implementada

### Camadas Implementadas

| Camada | Objetivo | Tecnologias | Particionamento |
|--------|----------|-------------|------------------|
| **BRONZE** | Snapshot bruto diário | ClickHouse MergeTree | `PARTITION BY ref_date` |
| **SILVER** | Normalização + Hash SHA-256 | ClickHouse MergeTree | `PARTITION BY ref_date` |
| **GOLD** | Deltas (I/U/D) | ClickHouse MergeTree | `PARTITION BY delta_date` |

### Características

- **Sem CDC**: Full extract diário
- **Idempotente**: Reprocessamento seguro
- **Hash por linha**: Comparação O(1)
- **Batch diário**: Previsível e escalável
- **Auditável**: Histórico completo no ClickHouse

### Casos de Uso

1. **Data Warehouse**: ClickHouse como fonte de verdade
2. **Auditoria**: Rastreamento completo de mudanças
3. **Reprocessamento**: Refazer qualquer dia sem risco
4. **Analytics**: Análise de evolução temporal
5. **Compliance**: Histórico imutável para regulamentação

---

## Próximos Passos

1. **Orquestração**: Airflow DAG para automação
2. **Alertas**: Notificações para anomalias (deletes em massa, etc)
3. **Múltiplas Tabelas**: Generalizar para todas as tabelas Oracle
4. **Otimização**: Tunning de partições e compressão
5. **Monitoramento**: Dashboards de volumetria e deltas